# Father — post-mortem investigation

**Premise.** Monitoring reported an unexplained `sshd` restart and an unrecognised session on this
host. The system was powered off; disk and memory were acquired. Nothing about the cause is assumed
here — the mechanism has to come out of the evidence.

**Scope.** The preserved disk and memory acquisitions of one run, plus the prepared extractions
derived from them.

**Boundary.** Scenario execution records guided the design of this lab and are disclosed in
Section 7. They are not evidence: every finding must cite an acquired disk, timeline or RAM record.

In [2]:
import json, os, re, sys
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

RUN_ID = "father-u22-20260913-01"

PROJECT_ROOT = next(
    p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (p / "shared" / "experiments").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))
from investigations.common import forensics as fx

case = fx.load_case(PROJECT_ROOT, RUN_ID)
prepared = json.loads((case.prepared / "prepare.json").read_text())

# Work from the case directory so every command below reads as a short, quotable path.
os.chdir(case.run_root)
IMG = str(case.disk_image.relative_to(case.run_root))
OUT, DATA = Path("investigation/output"), Path("investigation/data")
for d in (OUT, DATA):
    d.mkdir(parents=True, exist_ok=True)


def product(name):
    "Path of a prepared product, echoing the invocation that produced it."
    p = prepared["products"][name]
    root = str(case.run_root) + "/"
    print("$ " + " ".join(a.replace(root, "") for a in p["argv"]))
    print(f"# recorded: {p['state']}, exit {p.get('exit_code')}")
    return Path(p["path"])


def istat_times(text):
    "The four inode times from istat output, as UTC timestamps."
    labels = {"atime": "Accessed", "mtime": "File Modified",
              "ctime": "Inode Modified", "crtime": "File Created"}
    out = {}
    for key, label in labels.items():
        m = re.search(rf"^{label}:\s*(.+?)\s*$", text, re.M)
        out[key] = pd.to_datetime(re.sub(r"\s+\([A-Z]+\)$", "", m.group(1)), utc=True) if m else pd.NaT
    return out


SECTOR_SIZE = int(re.search(r"Units are in (\d+)-byte sectors",
                            Path(prepared["products"]["mmls"]["path"]).read_text()).group(1))
ROOT = [(int(p["argv"][p["argv"].index("-o") + 1]), name)
        for name, p in prepared["products"].items()
        if name.startswith("fsstat-") and p["state"] == "ok"
        and "File System Type: Ext4" in Path(p["path"]).read_text()]
print(f"Ext4 candidates: {ROOT}")
ROOT_OFFSET, ROOT_FS_PRODUCT = ROOT[0]

Ext4 candidates: [(227328, 'fsstat-006-offset-0000227328')]


## 0. Evidence and scope

### 0.1 What evidence is this, and when was it acquired?

In [3]:
print(f"{case.manifest['scenario']} on {case.platform['guest_os']}, "
      f"kernel {case.platform['kernel']} ({case.platform['arch']}), "
      f"recorded timezone {case.timezone}")

fx.show(pd.DataFrame([
    {"source": src, "path": rec["path"], "bytes": rec["size_bytes"],
     "recorded_sha256": rec["sha256"], "started_at": rec["started_at"],
     "ended_at": rec["ended_at"]}
    for src, rec in (("disk", case.acquisition["disk"]), ("ram", case.acquisition["memory"]))
]), n=2, caption="Acquired evidence")

father on Ubuntu 22.04.5 LTS, kernel 5.15.0-179-generic (x86_64), recorded timezone Etc/UTC


**Acquired evidence — showing 2 of 2 rows**

,source,path,bytes,recorded_sha256,started_at,ended_at
0,disk,disk/evidence_disk.E01,10737418240,ae591cfdfb26569b16478bbbc80bdd4e7f9bcc1741a84b13ebb9cb8b...,2026-09-13T20:46:38.398053Z,2026-09-13T20:47:16.034097Z
1,ram,memory/mem.raw,2147747795,e4089156b81a250dda36391932bf8011f70d0b54cd7acd4fcc859758...,2026-09-13T20:46:30.996505Z,2026-09-13T20:46:32.853530Z


### 0.2 Is the evidence internally consistent?

The EWF verification ran once during preparation and is not repeated here.

In [4]:
fx.sh(f"grep -E 'hash calculated|SUCCESS|FAILURE' {product('ewfverify')}",
      label="s0-02-ewfverify", out_dir=OUT)

$ /usr/local/bin/ewfverify -d sha256 dumps/disk/evidence_disk.E01
# recorded: ok, exit 0
$ grep -E 'hash calculated|SUCCESS|FAILURE' investigation/prepared/raw/ewfverify.txt
SHA256 hash calculated over data:	ae591cfdfb26569b16478bbbc80bdd4e7f9bcc1741a84b13ebb9cb8bdcb65aec
ewfverify: SUCCESS


CompletedProcess(args="grep -E 'hash calculated|SUCCESS|FAILURE' investigation/prepared/raw/ewfverify.txt", returncode=0, stdout='SHA256 hash calculated over data:\tae591cfdfb26569b16478bbbc80bdd4e7f9bcc1741a84b13ebb9cb8bdcb65aec\newfverify: SUCCESS\n', stderr='')

### 0.3 What limits this examination?

RAM and disk were acquired at different moments, so they are two states, not one snapshot. The
guest runs a vanilla logging profile, so absent records are bounded negatives rather than evidence
of absence.

In [5]:
RAM_ENDED = pd.to_datetime(case.acquisition["memory"]["ended_at"], utc=True)
DISK_STARTED = pd.to_datetime(case.acquisition["disk"]["started_at"], utc=True)
print(f"RAM capture ended  {RAM_ENDED.isoformat()}")
print(f"disk image started {DISK_STARTED.isoformat()}")
print(f"gap {(DISK_STARTED - RAM_ENDED).total_seconds():.3f} s — anything the guest wrote in that "
      f"window is on disk but not in memory")

RAM capture ended  2026-09-13T20:46:32.853530+00:00
disk image started 2026-09-13T20:46:38.398053+00:00
gap 5.545 s — anything the guest wrote in that window is on disk but not in memory


### 0.4 How should timestamps be read?

All output below is displayed in UTC.

In [6]:
tz_inode, _ = fx.resolve(IMG, ROOT_OFFSET, "/etc/timezone")
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {tz_inode}", label="s0-04-timezone", out_dir=OUT)

lt_inode, lt_chain = fx.resolve(IMG, ROOT_OFFSET, "/etc/localtime")
fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {lt_inode}",
      label="s0-04-localtime-istat", out_dir=OUT, tail=12)
print(f"/etc/localtime -> {lt_chain[-1]['path']}")
print(f"examiner timezone: {datetime.now().astimezone().tzinfo}")

$ icat -o 227328 dumps/disk/evidence_disk.E01 1290
Etc/UTC
$ istat -o 227328 -z UTC dumps/disk/evidence_disk.E01 33902
Flags: Extents, 
size: 114
num of links: 1

Inode Times:
Accessed:	2026-09-13 20:44:49.100000000 (UTC)
File Modified:	2026-03-19 14:04:40.000000000 (UTC)
Inode Modified:	2026-05-15 10:55:29.060828386 (UTC)
File Created:	2026-05-15 10:55:29.060828386 (UTC)

Direct Blocks:
511654 
/etc/localtime -> /usr/share/zoneinfo/Etc/UTC
examiner timezone: CEST


**Interpretation.** _(to write)_

## 1. Disk: what happened on this filesystem, and what persists?

### 1.1 Which partition holds the root filesystem?

In [7]:
fx.sh(f"cat {product('mmls')}", label="s1-01-mmls", out_dir=OUT)
fx.sh(f"head -n 30 {product(ROOT_FS_PRODUCT)}", label="s1-01-fsstat", out_dir=OUT)
print(f"root filesystem at sector {ROOT_OFFSET} ({SECTOR_SIZE}-byte sectors)")

$ /usr/local/bin/mmls dumps/disk/evidence_disk.E01
# recorded: ok, exit 0
$ cat investigation/prepared/raw/mmls.txt
GUID Partition Table (EFI)
Offset Sector: 0
Units are in 512-byte sectors

      Slot      Start        End          Length       Description
000:  Meta      0000000000   0000000000   0000000001   Safety Table
001:  -------   0000000000   0000002047   0000002048   Unallocated
002:  Meta      0000000001   0000000001   0000000001   GPT Header
003:  Meta      0000000002   0000000033   0000000032   Partition Table
004:  013       0000002048   0000010239   0000008192   
005:  014       0000010240   0000227327   0000217088   
006:  000       0000227328   0020971486   0020744159   
007:  -------   0020971487   0020971519   0000000033   Unallocated
$ /usr/local/bin/fsstat -o 0000227328 dumps/disk/evidence_disk.E01
# recorded: ok, exit 0
$ head -n 30 investigation/prepared/raw/fsstat-006-offset-0000227328.txt
FILE SYSTEM INFORMATION
--------------------------------------------
Fil

### 1.2 What changed on this filesystem shortly before acquisition?

No mechanism is assumed. The bodyfile is queried for every allocated object whose mtime, ctime or
crtime falls in the 24 hours before the disk was imaged, ordered by time. The window is a stated
choice, bounded by the acquisition time; a longer window is one edit away.

In [8]:
BODY = fx.load_bodyfile(product("allocated.body"))
ALLOCATED = BODY[BODY["name_state"].eq("allocated")]

WINDOW_START = DISK_STARTED - timedelta(hours=24)
in_window = ALLOCATED[
    ALLOCATED[["mtime_utc", "ctime_utc", "crtime_utc"]].ge(WINDOW_START).any(axis=1)
].copy()
in_window["last_change_utc"] = in_window[["mtime_utc", "ctime_utc", "crtime_utc"]].max(axis=1)
in_window = in_window.sort_values("last_change_utc")

in_window.to_csv(OUT / "s1-02-recent-changes.csv", index=False)
fx.show(in_window, cols=("last_change_utc", "name", "inode", "size", "mtime_utc",
                         "ctime_utc", "crtime_utc", "locator"),
        n=60, caption=f"Allocated objects changed after {WINDOW_START.isoformat()}")
print(f"{len(in_window)} objects in window; full result: {OUT}/s1-02-recent-changes.csv")

$ /usr/local/bin/fls -r -m / -o 227328 dumps/disk/evidence_disk.E01
# recorded: ok, exit 0


**Allocated objects changed after 2026-09-12T20:46:38.398053+00:00 — showing 60 of 82 rows**

,last_change_utc,name,inode,size,mtime_utc,ctime_utc,crtime_utc,locator
2192,2026-09-13 20:44:49+00:00,/tmp/.font-unix,258071,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2193
74476,2026-09-13 20:44:49+00:00,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/user-1...,74301,8388608,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-08-11 17:13:54+00:00,allocated.body#L74477
74475,2026-09-13 20:44:49+00:00,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system...,74157,8388608,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-08-11 17:13:43+00:00,allocated.body#L74476
2189,2026-09-13 20:44:49+00:00,/tmp/.X11-unix,258068,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2190
2190,2026-09-13 20:44:49+00:00,/tmp/.ICE-unix,258069,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2191
2191,2026-09-13 20:44:49+00:00,/tmp/.XIM-unix,258070,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2192
2193,2026-09-13 20:44:49+00:00,/tmp/.Test-unix,258072,4096,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,2026-09-13 20:44:49+00:00,allocated.body#L2194
74230,2026-09-13 20:44:50+00:00,/var/lib/cloud/instance -> /var/lib/cloud/instances/lab-...,74165,41,2026-09-13 20:44:50+00:00,2026-09-13 20:44:50+00:00,2026-09-13 20:44:50+00:00,allocated.body#L74231
74194,2026-09-13 20:44:53+00:00,/var/lib/cloud/instances/lab-ubuntu-22.04/scripts,74213,4096,2026-08-11 17:13:45+00:00,2026-09-13 20:44:53+00:00,2026-08-11 17:13:45+00:00,allocated.body#L74195
74193,2026-09-13 20:44:53+00:00,/var/lib/cloud/instances/lab-ubuntu-22.04/handlers,74212,4096,2026-08-11 17:13:45+00:00,2026-09-13 20:44:53+00:00,2026-08-11 17:13:45+00:00,allocated.body#L74194


82 objects in window; full result: investigation/output/s1-02-recent-changes.csv


**Interpretation.** _(to write — which of these are ordinary boot/shutdown activity, and which are
not? note anything whose timestamps sit near or after the acquisition times)_

### 1.3 Which persistence-relevant paths exist, and do any of them appear above?

Scope of this sweep, stated so the negative is bounded: the dynamic-loader preload file and
configuration directory, cron, systemd system units, rc links, profile scripts, and per-user shell
startup files. Anything outside this list was not examined here.

In [9]:
LOCATIONS = [
    ("dynamic loader preload",  r"^/etc/ld\.so\.preload$"),
    ("dynamic loader config",   r"^/etc/ld\.so\.conf\.d(?:/|$)"),
    ("cron",                    r"^/etc/(?:crontab$|cron(?:\.|/|$))"),
    ("systemd system units",    r"^/etc/systemd/system(?:/|$)"),
    ("rc links",                r"^/etc/rc[0-6S]\.d(?:/|$)"),
    ("profile scripts",         r"^/etc/profile\.d(?:/|$)"),
    ("shell startup",           r"^/(?:root|home/[^/]+)/\.(?:bashrc|profile)$"),
]
hits = pd.concat(
    [ALLOCATED[ALLOCATED["name"].str.match(p, na=False)].assign(location=name)
     for name, p in LOCATIONS],
    ignore_index=True,
)
hits["in_window"] = hits["locator"].isin(in_window["locator"])
hits.to_csv(OUT / "s1-03-persistence-paths.csv", index=False)

print(hits.groupby("location").agg(entries=("name", "size"),
                                   changed_in_window=("in_window", "sum")).to_string())
print(f"\nfull inventory: {OUT}/s1-03-persistence-paths.csv\n")

fx.show(hits[hits["in_window"]].sort_values("mtime_utc", ascending=False),
        cols=("location", "name", "inode", "mode", "size", "mtime_utc"),
        n=20, caption="Persistence-location entries that changed inside the window")

preload = hits[hits["location"].eq("dynamic loader preload")]
print(f"preload entries found: {len(preload)}")
PRELOAD_ROW = preload.iloc[0]
PRELOAD_PATH = str(PRELOAD_ROW["name"])

                        entries  changed_in_window
location                                          
cron                         18                  0
dynamic loader config         3                  0
dynamic loader preload        1                  1
profile scripts               9                  0
rc links                    102                  0
shell startup                 4                  0
systemd system units        119                  0

full inventory: investigation/output/s1-03-persistence-paths.csv



**Persistence-location entries that changed inside the window — showing 1 of 1 rows**

,location,name,inode,mode,size,mtime_utc
0,dynamic loader preload,/etc/ld.so.preload,74252,r/rrw-r--r--,17,2026-09-13 20:46:32+00:00


preload entries found: 1


**Interpretation.** _(to write — a stock Ubuntu cloud image ships no `/etc/ld.so.preload`)_

### 1.4 What does that configuration file contain?

In [10]:
preload_inode, _ = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_PATH)
preload_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {preload_inode}",
                      label="s1-04-preload-istat", out_dir=OUT, tail=20)
preload_icat = fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {preload_inode}",
                     label="s1-04-preload-icat", out_dir=OUT)

ENTRIES = [w for line in preload_icat.stdout.splitlines()
           for w in line.partition("#")[0].split()]
print(f"\nconfiguration entries: {ENTRIES}")
PRELOAD_OBJECT_PATH = ENTRIES[0]

$ istat -o 227328 -z UTC dumps/disk/evidence_disk.E01 74252
inode: 74252
Allocated
Group: 4
Generation Id: 3702835834
uid / gid: 0 / 0
mode: rrw-r--r--
Flags: Extents, 
size: 17
num of links: 1

Inode Times:
Accessed:	2026-09-13 20:46:32.884000000 (UTC)
File Modified:	2026-09-13 20:46:32.880000000 (UTC)
Inode Modified:	2026-09-13 20:46:32.880000000 (UTC)
File Created:	2026-09-13 20:46:32.880000000 (UTC)

Direct Blocks:
296191 
$ icat -o 227328 dumps/disk/evidence_disk.E01 74252
/lib/selinux.so.3
configuration entries: ['/lib/selinux.so.3']


**Interpretation.** _(to write)_

### 1.5 What is the referenced object?

In [11]:
object_inode, object_chain = fx.resolve(IMG, ROOT_OFFSET, PRELOAD_OBJECT_PATH)
print(" -> ".join(f"{e['path']}({e['inode']})" + (f" [link {e['target']}]" if "target" in e else "")
                  for e in object_chain))

object_istat = fx.sh(f"istat -o {ROOT_OFFSET} -z UTC {IMG} {object_inode}",
                     label="s1-05-object-istat", out_dir=OUT, tail=20)

OBJECT_BIN = DATA / "s1-05-referenced-object.bin"
fx.sh(f"icat -o {ROOT_OFFSET} {IMG} {object_inode}",
      label=OBJECT_BIN.name, out_dir=DATA, binary=True, show=False)

fx.sh(f"file {OBJECT_BIN}", label="s1-05-object-file", out_dir=OUT)
fx.sh(f"sha256sum {OBJECT_BIN}", label="s1-05-object-sha256", out_dir=OUT)
fx.sh(f"readelf -h -d {OBJECT_BIN}", label="s1-05-object-readelf", out_dir=OUT, tail=20)
strings = fx.sh(f"strings -a -t x {OBJECT_BIN}", label="s1-05-object-strings",
                out_dir=OUT, tail=0)
print(f"\n{len(strings.stdout.splitlines())} strings preserved in {OUT}/s1-05-object-strings.txt")

/lib(1542) [link usr/lib] -> /usr(1582) -> /usr/lib(2938) -> /usr/lib/selinux.so.3(74253)
$ istat -o 227328 -z UTC dumps/disk/evidence_disk.E01 74253
inode: 74253
Allocated
Group: 4
Generation Id: 33261919
uid / gid: 0 / 0
mode: rrw-r--r--
Flags: Extents, 
size: 32784
num of links: 1

Inode Times:
Accessed:	2026-09-13 20:46:00.416000000 (UTC)
File Modified:	2026-01-30 08:20:56.000000000 (UTC)
Inode Modified:	2026-09-13 20:45:24.340000000 (UTC)
File Created:	2026-09-13 20:45:24.328000000 (UTC)

Direct Blocks:
338478 338479 338480 338481 338482 338483 338484 338485 
338486 
$ file investigation/data/s1-05-referenced-object.bin
investigation/data/s1-05-referenced-object.bin: ELF 64-bit LSB shared object, x86-64, version 1 (SYSV), dynamically linked, BuildID[sha1]=96daef8bb7ab389abcb5aa9458436759949849c7, not stripped
$ sha256sum investigation/data/s1-05-referenced-object.bin
87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711  investigation/data/s1-05-referenced-object.bin
$ 

## 2. Surrounding activity: who, when, and what else changed?

The complete outputs from 1.2 and 1.5 are the pivot source. Account and time values come from
2.1; object strings are limited to recovered paths, authentication text and concealment markers.
Routine ELF symbols and scenario-supplied names are not search terms.

### 2.1 Which login sessions exist?

In [12]:
SESSION_OUTPUTS = {"wtmp": "", "btmp": ""}
for name, reader in (("wtmp", "last"), ("btmp", "lastb")):
    guest_path = f"/var/log/{name}"
    rows = ALLOCATED[ALLOCATED["name"].eq(guest_path)]
    print(f"{guest_path}: {len(rows)} allocated bodyfile entries")
    if len(rows) != 1:
        continue
    inode = str(rows.iloc[0]["inode"])
    target = DATA / f"s2-01-{name}"
    extracted = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(guest_path)} {fx.q(IMG)} > {fx.q(target)}",
                      label=f"s2-01-{name}-fcat", out_dir=OUT)
    print(f"{guest_path} (inode {inode}): fcat exit {extracted.returncode}, {target.stat().st_size} bytes")
    if extracted.returncode != 0:
        continue
    viewed = fx.sh(f"TZ=UTC {reader} -F -i -f {fx.q(target)}",
                   label=f"s2-01-{reader}", out_dir=OUT, tail=40)
    SESSION_OUTPUTS[name] = viewed.stdout

excluded = {"reboot", "shutdown", "runlevel"}
session_lines = [line for text in SESSION_OUTPUTS.values() for line in text.splitlines()
                 if line.split() and " begins " not in line and line.split()[0] not in excluded]
SESSION_USERS = sorted({line.split()[0] for line in session_lines})
stamps = re.findall(r"(?:Mon|Tue|Wed|Thu|Fri|Sat|Sun) \w{3} +\d+ \d{2}:\d{2}:\d{2} \d{4}",
                    "\n".join(session_lines))
SESSION_TIMES = sorted({pd.to_datetime(stamp, utc=True) for stamp in stamps})
TIME_PIVOTS = sorted({v for stamp in SESSION_TIMES
                      for v in (stamp.strftime("%b %e %H:%M"),
                                stamp.strftime("%Y-%m-%dT%H:%M"))})
print(f"session users: {SESSION_USERS}")
print(f"session time pivots: {TIME_PIVOTS}")

/var/log/wtmp: 1 allocated bodyfile entries
$ fcat -o 227328 /var/log/wtmp dumps/disk/evidence_disk.E01 > investigation/data/s2-01-wtmp
/var/log/wtmp (inode 73185): fcat exit 0, 8448 bytes
$ TZ=UTC last -F -i -f investigation/data/s2-01-wtmp
labuser  pts/0        192.168.100.1    Sun Sep 13 20:44:59 2026 - Sun Sep 13 20:46:30 2026  (00:01)
reboot   system boot  0.0.0.0          Sun Sep 13 20:44:48 2026 - Sun Sep 13 20:46:35 2026  (00:01)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13:58 2026 - Tue Aug 11 17:13:58 2026  (00:00)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13:57 2026 - Tue Aug 11 17:13:58 2026  (00:00)
labuser  pts/0        192.168.100.1    Tue Aug 11 17:13:56 2026 - Tue Aug 11 17:13:57 2026  (00:00)
reboot   system boot  0.0.0.0          Tue Aug 11 17:13:43 2026 - Tue Aug 11 17:14:01 2026  (00:00)

s2-01-wtmp begins Tue Aug 11 17:13:43 2026
/var/log/btmp: 1 allocated bodyfile entries
$ fcat -o 227328 /var/log/btmp dumps/disk/evidence_disk.E01 > investig

**Interpretation.** _(to write)_

### 2.2 What local accounts and privileges existed?

The shadow file is extracted for later byte comparison, but only its inode metadata is displayed.

In [13]:
ACCOUNT_FILES = {}
for guest_path in ("/etc/passwd", "/etc/group", "/etc/shadow"):
    inode, _ = fx.resolve(IMG, ROOT_OFFSET, guest_path)
    target = DATA / f"s2-02-{Path(guest_path).name}"
    extracted = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(guest_path)} {fx.q(IMG)} > {fx.q(target)}",
                      label=f"s2-02-{target.name}-fcat", out_dir=OUT, tail=10)
    ACCOUNT_FILES[guest_path] = target
    print(f"{guest_path}: fcat exit {extracted.returncode}, {target.stat().st_size} bytes")
    if guest_path != "/etc/shadow":
        fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-02-{target.name}-numbered",
              out_dir=OUT, tail=80)

shadow_inode, _ = fx.resolve(IMG, ROOT_OFFSET, "/etc/shadow")
fx.sh(f"istat -o {fx.q(ROOT_OFFSET)} -z UTC {fx.q(IMG)} {fx.q(shadow_inode)}",
      label="s2-02-shadow-istat", out_dir=OUT, tail=20)
sudo_rows = ALLOCATED[ALLOCATED["name"].str.match(r"^/etc/sudoers(?:$|\.d(?:/|$))", na=False)]
fx.show(sudo_rows, cols=("name", "inode", "mode", "uid", "gid", "size", "locator"),
        n=40, caption="Sudo policy inventory")
for row in sudo_rows[sudo_rows["mode"].str.startswith("r/")].itertuples():
    target = DATA / f"s2-02-sudo-{row.inode}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-02-sudo-{row.inode}-fcat", out_dir=OUT, tail=10)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-02-sudo-{row.inode}-numbered",
          out_dir=OUT, tail=120)

$ fcat -o 227328 /etc/passwd dumps/disk/evidence_disk.E01 > investigation/data/s2-02-passwd
/etc/passwd: fcat exit 0, 1762 bytes
$ nl -ba investigation/data/s2-02-passwd
     1	root:x:0:0:root:/root:/bin/bash
     2	daemon:x:1:1:daemon:/usr/sbin:/usr/sbin/nologin
     3	bin:x:2:2:bin:/bin:/usr/sbin/nologin
     4	sys:x:3:3:sys:/dev:/usr/sbin/nologin
     5	sync:x:4:65534:sync:/bin:/bin/sync
     6	games:x:5:60:games:/usr/games:/usr/sbin/nologin
     7	man:x:6:12:man:/var/cache/man:/usr/sbin/nologin
     8	lp:x:7:7:lp:/var/spool/lpd:/usr/sbin/nologin
     9	mail:x:8:8:mail:/var/mail:/usr/sbin/nologin
    10	news:x:9:9:news:/var/spool/news:/usr/sbin/nologin
    11	uucp:x:10:10:uucp:/var/spool/uucp:/usr/sbin/nologin
    12	proxy:x:13:13:proxy:/bin:/usr/sbin/nologin
    13	www-data:x:33:33:www-data:/var/www:/usr/sbin/nologin
    14	backup:x:34:34:backup:/var/backups:/usr/sbin/nologin
    15	list:x:38:38:Mailing List Manager:/var/list:/usr/sbin/nologin
    16	irc:x:39:39:ircd:/run/ircd:/usr

**Sudo policy inventory — showing 4 of 4 rows**

,name,inode,mode,uid,gid,size,locator
1585,/etc/sudoers,1006,r/rr--r-----,0,0,1671,allocated.body#L1586
1586,/etc/sudoers.d,1007,d/drwxr-x---,0,0,4096,allocated.body#L1587
1587,/etc/sudoers.d/README,1008,r/rr--r-----,0,0,1096,allocated.body#L1588
1588,/etc/sudoers.d/90-cloud-init-users,618,r/rr--r-----,0,0,141,allocated.body#L1589


$ fcat -o 227328 /etc/sudoers dumps/disk/evidence_disk.E01 > investigation/data/s2-02-sudo-1006
/etc/sudoers: fcat exit 0, 1671 bytes
$ nl -ba investigation/data/s2-02-sudo-1006
     1	#
     2	# This file MUST be edited with the 'visudo' command as root.
     3	#
     4	# Please consider adding local content in /etc/sudoers.d/ instead of
     5	# directly modifying this file.
     6	#
     7	# See the man page for details on how to write a sudoers file.
     8	#
     9	Defaults	env_reset
    10	Defaults	mail_badpass
    11	Defaults	secure_path="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/snap/bin"
    12	Defaults	use_pty
    13	
    14	# This preserves proxy settings from user environments of root
    15	# equivalent users (group sudo)
    16	#Defaults:%sudo env_keep += "http_proxy https_proxy ftp_proxy all_proxy no_proxy"
    17	
    18	# This allows running arbitrary commands, but so does ALL, and it means
    19	# different sudoers have their choice of editor resp

**Interpretation.** _(to write)_

### 2.3 What do the authentication and service logs record?

The search terms are printed before use. They come from the sessions above, the initial `sshd`
trigger, and the already-enumerated object strings in 1.5.

In [14]:
# Analyst selection, read from investigation/output/s1-05-object-strings.txt.
# Run-specific by nature: a different run needs this list re-read and rewritten.
SELECTED_STRINGS = [
    "AUTHENTICATE:", "lobster", "Enjoy the shell!", "__malicious_",
    "/proc/net/tcp", "/tmp/silly.txt", PRELOAD_PATH, PRELOAD_OBJECT_PATH,
]
SEARCH_PIVOTS = sorted(set(SESSION_USERS + ["sshd"]
                          + [s for s in SELECTED_STRINGS if s in strings.stdout]))
PIVOT_FILE = OUT / "s2-03-pivots.txt"
PIVOT_FILE.write_text("\n".join(SEARCH_PIVOTS) + "\n")
print("search pivots:\n" + "\n".join(f"  {v}" for v in SEARCH_PIVOTS))

log_rows = ALLOCATED[ALLOCATED["name"].str.match(
    r"^/var/log/(?:auth\.log$|syslog$|journal/.+\.journal$)", na=False)]
fx.show(log_rows, cols=("name", "inode", "size", "mtime_utc", "locator"),
        n=20, caption="Allocated logs selected for extraction")
LOG_FILES = {}
for row in log_rows.itertuples():
    target = DATA / f"s2-03-{row.inode}-{Path(row.name).name}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-03-{row.inode}-fcat", out_dir=OUT)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode == 0:
        LOG_FILES[row.name] = target

journal_files = [t for n, t in LOG_FILES.items() if n.endswith(".journal")]
print(f"journal files extracted: {len(journal_files)}")
LOG_SOURCES = {Path(name).name: target for name, target in LOG_FILES.items() if name in ("/var/log/auth.log", "/var/log/syslog")}
if journal_files:
    files = " ".join(f"--file={fx.q(path)}" for path in journal_files)
    rendered = fx.sh(f"journalctl --utc --no-pager -o short-iso-precise {files}",
                     label="s2-03-journal-rendered", out_dir=OUT, tail=0)
    print(f"journalctl exit {rendered.returncode}, {len(rendered.stdout.splitlines())} rendered lines")
    if rendered.returncode == 0:
        LOG_SOURCES["journal"] = OUT / "s2-03-journal-rendered.txt"
if (latest := max(SESSION_TIMES, default=None)) is not None:
    print(f"search day from 2.1: {latest.date().isoformat()} UTC")
    for name, source in LOG_SOURCES.items():
        day = latest.strftime("%Y-%m-%d" if name == "journal" else "%b %e")
        matched = fx.sh(f"grep -nF {fx.q(day)} {fx.q(source)} | grep -Fi -f {fx.q(PIVOT_FILE)}",
                        label=f"s2-03-{name}-matches", out_dir=OUT, tail=80)
        print(f"{name}: {len(matched.stdout.splitlines())} matching lines "
              f"(grep exit {matched.returncode})")
else:
    print("log search not run: 2.1 supplied zero session times")

search pivots:
  /etc/ld.so.preload
  /lib/selinux.so.3
  /proc/net/tcp
  /tmp/silly.txt
  AUTHENTICATE:
  Enjoy the shell!
  __malicious_
  labuser
  lobster
  sshd


**Allocated logs selected for extraction — showing 6 of 6 rows**

,name,inode,size,mtime_utc,locator
74473,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system...,74170,8388608,2026-09-13 20:46:35+00:00,allocated.body#L74474
74474,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/user-1...,74172,8388608,2026-09-13 20:46:35+00:00,allocated.body#L74475
74475,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system...,74157,8388608,2026-09-13 20:44:49+00:00,allocated.body#L74476
74476,/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/user-1...,74301,8388608,2026-09-13 20:44:49+00:00,allocated.body#L74477
74494,/var/log/auth.log,74263,8505,2026-09-13 20:46:32+00:00,allocated.body#L74495
74495,/var/log/syslog,74294,269086,2026-09-13 20:46:32+00:00,allocated.body#L74496


$ fcat -o 227328 /var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system.journal dumps/disk/evidence_disk.E01 > investigation/data/s2-03-74170-system.journal
/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system.journal: fcat exit 0, 8388608 bytes
$ fcat -o 227328 /var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/user-1000.journal dumps/disk/evidence_disk.E01 > investigation/data/s2-03-74172-user-1000.journal
/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/user-1000.journal: fcat exit 0, 8388608 bytes
$ fcat -o 227328 /var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system@487c9cf910684eeebc996a06b5f9917e-0000000000000001-000658c897cd7459.journal dumps/disk/evidence_disk.E01 > investigation/data/s2-03-74157-system@487c9cf910684eeebc996a06b5f9917e-0000000000000001-000658c897cd7459.journal
/var/log/journal/5197e2566dcb4cd799bf2ba16efa2ba1/system@487c9cf910684eeebc996a06b5f9917e-0000000000000001-000658c897cd7459.journal: fcat exit 0, 8388608 bytes
$ fcat -o 227328 /var/log/journal/

**Interpretation.** _(to write)_

### 2.4 Is there shell history for any account?

Scope: allocated bodyfile entries named `.*history` directly under `/root` or an immediate
`/home/<user>` directory. A zero-row result is limited to those names and allocated entries; it
does not establish erasure.

In [15]:
history_rows = ALLOCATED[ALLOCATED["name"].str.match(
    r"^/(?:root|home/[^/]+)/\.[^/]*history$", na=False
)]
print("scope searched: allocated.body; /root and immediate /home/<user>; names .*[Hh]istory")
fx.show(history_rows, cols=("name", "inode", "mode", "uid", "gid",
                             "size", "mtime_utc", "locator"),
        n=30, caption="Allocated shell-history candidates")
for row in history_rows.itertuples():
    target = DATA / f"s2-04-history-{row.inode}"
    got = fx.sh(f"fcat -o {fx.q(ROOT_OFFSET)} {fx.q(row.name)} {fx.q(IMG)} > {fx.q(target)}",
                label=f"s2-04-history-{row.inode}-fcat", out_dir=OUT, tail=10)
    print(f"{row.name}: fcat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode == 0:
        fx.sh(f"nl -ba {fx.q(target)}", label=f"s2-04-history-{row.inode}-numbered",
              out_dir=OUT, tail=80)

scope searched: allocated.body; /root and immediate /home/<user>; names .*[Hh]istory


**Allocated shell-history candidates — showing 0 of 0 rows**

,name,inode,mode,uid,gid,size,mtime_utc,locator


**Interpretation.** _(to write)_

### 2.5 Do the staging paths named in the logs exist, and what else is there?

The log lines recovered in 2.3 name specific paths written during the session. This block does
not assume where staging happened: it resolves each directory from the evidence and inventories
it in full — allocated and deleted separately — so that the log-named paths can be confirmed on
disk and anything the logs did **not** mention is also visible. All four listings are displayed
before anything is selected.

In [16]:
STAGING_LISTINGS = {}
for guest_path in ("/tmp", "/dev/shm"):
    try:
        directory_inode, chain = fx.resolve(IMG, ROOT_OFFSET, guest_path)
    except fx.ResolveError as error:
        print(f"{guest_path}: not resolved; 0 inventories ({error})")
        continue
    print(f"{guest_path}: " + " -> ".join(
        f"{entry['path']}({entry['inode']})" +
        (f" [link {entry['target']}]" if "target" in entry else "")
        for entry in chain))
    for state, switch in (("allocated", "-u"), ("deleted", "-d")):
        label = f"s2-05-{guest_path.strip('/').replace('/', '-')}-{state}"
        listing = fx.sh(
            f"fls -r -p {switch} -o {fx.q(ROOT_OFFSET)} {fx.q(IMG)} {fx.q(directory_inode)}",
            label=label, out_dir=OUT, tail=80)
        STAGING_LISTINGS[(guest_path, state)] = listing.stdout
        print(f"{guest_path} {state}: {len(listing.stdout.splitlines())} entries, "
              f"fls exit {listing.returncode}")

/tmp: /tmp(1581)
$ fls -r -p -u -o 227328 dumps/disk/evidence_disk.E01 1581
d/d 258067:	snap-private-tmp
d/d 258113:	snap-private-tmp/snap.lxd
d/d 258114:	snap-private-tmp/snap.lxd/tmp
d/d 258068:	.X11-unix
d/d 258069:	.ICE-unix
d/d 258070:	.XIM-unix
d/d 258071:	.font-unix
d/d 258072:	.Test-unix
r/r 74173:	__malicious_recon
r/r 74163:	__malicious_harvest
/tmp allocated: 10 entries, fls exit 0
$ fls -r -p -d -o 227328 dumps/disk/evidence_disk.E01 1581
/tmp deleted: 0 entries, fls exit 0
/dev/shm: /dev(26) -> /dev/shm(35)
$ fls -r -p -u -o 227328 dumps/disk/evidence_disk.E01 35
/dev/shm allocated: 0 entries, fls exit 0
$ fls -r -p -d -o 227328 dumps/disk/evidence_disk.E01 35
/dev/shm deleted: 0 entries, fls exit 0


**Interpretation.** _(to write)_

### 2.6 What are the staged files, and do any duplicate a system file?

Every regular file in the four inventories above is selected. `icat` preserves its bytes;
`file`, `sha256sum` and `strings` characterise it. Hash equality is checked against the extracted
account databases from 2.2; decoded text is never compared.

In [17]:
fls_line = re.compile(
    r"^(?P<kind>\S+)\s+(?:\*\s*)?(?P<inode>\d+(?:-\d+-\d+)?)(?:\(realloc\))?:\s+(?P<name>.+)$")
staged = []
for (guest_path, state), text in STAGING_LISTINGS.items():
    for line in text.splitlines():
        match = fls_line.match(line)
        if match and match["kind"].startswith("r/"):
            staged.append({"directory": guest_path, "state": state, **match.groupdict()})
fx.show(pd.DataFrame(staged), n=40, caption="Regular files selected from the inventories")
original_hashes = {name: fx.sha256_file(path) for name, path in ACCOUNT_FILES.items()}
for index, item in enumerate(staged, 1):
    target = DATA / f"s2-06-{index:02d}-{item['state']}.bin"
    got = fx.sh(f"icat -o {fx.q(ROOT_OFFSET)} {fx.q(IMG)} {fx.q(item['inode'])} > {fx.q(target)}",
                label=f"s2-06-{index:02d}-icat", out_dir=OUT, tail=10)
    print(f"{item['directory']}/{item['name']} [{item['state']} inode {item['inode']}]: "
          f"icat exit {got.returncode}, {target.stat().st_size} bytes")
    if got.returncode != 0:
        continue
    fx.sh(f"file {fx.q(target)}", label=f"s2-06-{index:02d}-file", out_dir=OUT)
    fx.sh(f"sha256sum {fx.q(target)}", label=f"s2-06-{index:02d}-sha256", out_dir=OUT)
    fx.sh(f"strings -a -n 6 {fx.q(target)}", label=f"s2-06-{index:02d}-strings",
          out_dir=OUT, tail=40)
    digest = fx.sha256_file(target)
    matches = [n for n, d in original_hashes.items() if d == digest]
    print(f"byte-identical extracted system files: {matches or 'none'}")
    for name in matches:
        fx.sh(f"sha256sum {fx.q(target)} {fx.q(ACCOUNT_FILES[name])}",
              label=f"s2-06-{index:02d}-comparison", out_dir=OUT)

**Regular files selected from the inventories — showing 2 of 2 rows**

,directory,state,kind,inode,name
0,/tmp,allocated,r/r,74173,__malicious_recon
1,/tmp,allocated,r/r,74163,__malicious_harvest


$ icat -o 227328 dumps/disk/evidence_disk.E01 74173 > investigation/data/s2-06-01-allocated.bin
/tmp/__malicious_recon [allocated inode 74173]: icat exit 0, 2328 bytes
$ file investigation/data/s2-06-01-allocated.bin
investigation/data/s2-06-01-allocated.bin: ASCII text
$ sha256sum investigation/data/s2-06-01-allocated.bin
41023383d34170ac4e96a5105d55e2ad0e1f1f5815281170b8b75a003845b9dc  investigation/data/s2-06-01-allocated.bin
$ strings -a -n 6 investigation/data/s2-06-01-allocated.bin
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
root:x:0:0:root:/root:/bin/bash
daemon:x:1:1:daemon:/usr/sbin:/usr/sbin/nologin
bin:x:2:2:bin:/bin:/usr/sbin/nologin
sys:x:3:3:sys:/dev:/usr/sbin/nologin
sync:x:4:65534:sync:/bin:/bin/sync
games:x:5:60:games:/usr/games:/usr/sbin/nologin
man:x:6:12:ma

**Interpretation.** _(to write)_

## 3. Memory: what was running?

The prepared RAM products are loaded from their preserved JSON. Plugins absent from preparation
run inline against the acquired memory image and the recorded symbol directory.

### 3.1 Does the image match the symbols?

In [18]:
MEM = os.path.relpath(case.memory_image, case.run_root)
ISF_DIR = os.path.relpath(case.isf_path.parent, case.run_root)
VOL_BASE = f"vol3 -f {fx.q(MEM)} -s {fx.q(ISF_DIR)} -r json"
BANNERS = fx.load_vol(product("banners.json"))
fx.show(BANNERS, cols=("Banner", "Offset", "locator"),
        n=len(BANNERS), caption="Prepared Linux banners")

$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json banners.Banners
# recorded: ok, exit 0


**Prepared Linux banners — showing 3 of 3 rows**

,Banner,Offset,locator
0,Linux version 5.15.0-179-generic (buildd@lcy02-amd64-040...,885762888,banners.json#/0
1,Linux version 5.15.0-179-generic (buildd@lcy02-amd64-040...,2011169280,banners.json#/1
2,Linux version 5.15.0-179-generic (buildd@lcy02-amd64-040...,2044883832,banners.json#/2


**Interpretation.** _(to write)_

### 3.2 What processes existed, and how are they related?

In [19]:
PSTREE = fx.load_vol(product("pstree.json"))
PSAUX = fx.load_vol(product("psaux.json"))
fx.show(PSTREE, cols=("PID", "PPID", "TID", "COMM",
                            "parent_locator", "locator"),
        n=50, caption="Prepared process tree")
fx.show(PSAUX, cols=("PID", "PPID", "COMM", "ARGS", "locator"),
        n=50, caption="Prepared process arguments")
print(f"pstree rows: {len(PSTREE)}; psaux rows: {len(PSAUX)}")

$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json linux.pstree.PsTree
# recorded: ok, exit 0
$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json linux.psaux.PsAux
# recorded: ok, exit 0


**Prepared process tree — showing 50 of 129 rows**

,PID,PPID,TID,COMM,parent_locator,locator
0,1,0,1,systemd,NaN,pstree.json#/0
1,391,1,391,systemd-journal,pstree.json#/0,pstree.json#/0/__children/0
2,428,1,428,multipathd,pstree.json#/0,pstree.json#/0/__children/1
3,431,1,431,systemd-udevd,pstree.json#/0,pstree.json#/0/__children/2
4,504,1,504,systemd-timesyn,pstree.json#/0,pstree.json#/0/__children/3
5,569,1,569,systemd-network,pstree.json#/0,pstree.json#/0/__children/4
6,571,1,571,systemd-resolve,pstree.json#/0,pstree.json#/0/__children/5
7,604,1,604,cron,pstree.json#/0,pstree.json#/0/__children/6
8,605,1,605,dbus-daemon,pstree.json#/0,pstree.json#/0/__children/7
9,612,1,612,irqbalance,pstree.json#/0,pstree.json#/0/__children/8


**Prepared process arguments — showing 50 of 129 rows**

,PID,PPID,COMM,ARGS,locator
0,1,0,systemd,/sbin/init,psaux.json#/0
1,2,0,kthreadd,[kthreadd],psaux.json#/1
2,3,2,rcu_gp,[rcu_gp],psaux.json#/2
3,4,2,rcu_par_gp,[rcu_par_gp],psaux.json#/3
4,5,2,slub_flushwq,[slub_flushwq],psaux.json#/4
5,6,2,netns,[netns],psaux.json#/5
6,7,2,kworker/0:0,[kworker/0:0],psaux.json#/6
7,8,2,kworker/0:0H,[kworker/0:0H],psaux.json#/7
8,9,2,kworker/u4:0,[kworker/u4:0],psaux.json#/8
9,10,2,mm_percpu_wq,[mm_percpu_wq],psaux.json#/9


pstree rows: 129; psaux rows: 129


**Interpretation.** _(to write)_

### 3.3 Which processes map the object found in Section 1?

The filter is the final evidence-resolved path in `object_chain`, not a typed filename. Counts
use distinct PID and path pairs rather than individual virtual-memory mapping rows.

In [20]:
MAPS = fx.load_vol(product("proc.Maps.json"))
OBJECT_MAP_PATH = object_chain[-1]["path"]
OBJECT_MAPS = MAPS[MAPS["File Path"].eq(OBJECT_MAP_PATH)].copy()
OBJECT_PAIRS = (OBJECT_MAPS[["PID", "Process", "File Path"]]
                .drop_duplicates(["PID", "File Path"])
                .sort_values(["PID", "File Path"]))
MAPPING_PIDS = OBJECT_PAIRS["PID"].dropna().astype(int).tolist()
print(f"Section 1 path: {PRELOAD_OBJECT_PATH}")
print(f"resolved map path: {OBJECT_MAP_PATH}; disk inode: {object_inode}")
fx.show(OBJECT_MAPS, cols=("PID", "Process", "Start", "End", "Flags",
                                  "Inode", "File Path", "locator"),
        n=40, caption="Mapping rows for the resolved object path")
fx.show(OBJECT_PAIRS, n=40, caption="Distinct PID and object-path pairs")
print(f"mapping rows: {len(OBJECT_MAPS)}; distinct PID + path pairs: "
      f"{len(OBJECT_PAIRS)}; PIDs: {MAPPING_PIDS}")

$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json linux.proc.Maps
# recorded: ok, exit 0
Section 1 path: /lib/selinux.so.3
resolved map path: /usr/lib/selinux.so.3; disk inode: 74253


**Mapping rows for the resolved object path — showing 10 of 10 rows**

,PID,Process,Start,End,Flags,Inode,File Path,locator
3875,1087,sshd,140178528165888,140178528174080,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3875
3876,1087,sshd,140178528174080,140178528186368,r-x,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3876
3877,1087,sshd,140178528186368,140178528190464,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3877
3878,1087,sshd,140178528190464,140178528194560,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3878
3879,1087,sshd,140178528194560,140178528198656,rw-,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3879
3904,1089,sh,139722316402688,139722316410880,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3904
3905,1089,sh,139722316410880,139722316423168,r-x,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3905
3906,1089,sh,139722316423168,139722316427264,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3906
3907,1089,sh,139722316427264,139722316431360,r--,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3907
3908,1089,sh,139722316431360,139722316435456,rw-,74253,/usr/lib/selinux.so.3,proc.Maps.json#/3908


**Distinct PID and object-path pairs — showing 2 of 2 rows**

,PID,Process,File Path
3875,1087,sshd,/usr/lib/selinux.so.3
3904,1089,sh,/usr/lib/selinux.so.3


mapping rows: 10; distinct PID + path pairs: 2; PIDs: [1087, 1089]


**Interpretation.** _(to write)_

### 3.4 Is `LD_PRELOAD` present in any process environment?

A zero-row result is expected for file-based preload configuration and is reported as a result.

In [21]:
ENVARS_JSON = OUT / "s3-04-envars.json"
envars_run = fx.sh(f"{VOL_BASE} linux.envars.Envars > {fx.q(ENVARS_JSON)}",
                    label="s3-04-envars-run", out_dir=OUT, tail=20)
print(f"linux.envars exit {envars_run.returncode}")
ENVARS = pd.DataFrame()
ENV_PRELOAD = pd.DataFrame()
if envars_run.returncode == 0:
    ENVARS = fx.load_vol(ENVARS_JSON)
    if not ENVARS.empty:
        env_text = ENVARS.astype("string")
        mask = env_text.apply(
            lambda col: col.str.contains("LD_PRELOAD", regex=False, na=False)
        ).any(axis=1)
        ENV_PRELOAD = ENVARS[mask]
    fx.show(ENV_PRELOAD, n=40, caption="Environment rows containing LD_PRELOAD")
    print(f"environment rows: {len(ENVARS)}; LD_PRELOAD rows: {len(ENV_PRELOAD)}")

$ vol3 -f dumps/memory/mem.raw -s ../../isf -r json linux.envars.Envars > investigation/output/s3-04-envars.json



Progress:   43.36		Scanning Elf64Layer using BytesScanner

Progress:   43.75		Scanning Elf64Layer using BytesScanner

Progress:   44.14		Scanning Elf64Layer using BytesScanner

Progress:   44.53		Scanning Elf64Layer using BytesScanner

Progress:   44.92		Scanning Elf64Layer using BytesScanner

Progress:   45.31		Scanning Elf64Layer using BytesScanner

Progress:   45.70		Scanning Elf64Layer using BytesScanner

Progress:   46.09		Scanning Elf64Layer using BytesScanner

Progress:   46.48		Scanning Elf64Layer using BytesScanner

Progress:  100.00		Stacking attempts finished            


linux.envars exit 0


**Environment rows containing LD_PRELOAD — showing 0 of 0 rows**

,COMM,KEY,PID,PPID,VALUE,parent_locator,locator


environment rows: 184; LD_PRELOAD rows: 0


**Interpretation.** _(to write)_

### 3.5 Which endpoints were open at capture?

The prepared socket and open-file inventories are displayed in full before attribution. The
object string `/proc/net/tcp` came from 1.5: a connection hidden from the live host may still
appear in memory-resident kernel structures.

In [22]:
SOCKSTAT = fx.load_vol(product("sockstat.json"))
LSOF = fx.load_vol(product("lsof.json"))
with pd.option_context("display.max_rows", None):
    fx.show(SOCKSTAT, n=len(SOCKSTAT), caption="Full prepared socket inventory")
    fx.show(LSOF, n=len(LSOF), caption="Full prepared open-file inventory")
MAPPED_SOCKETS = SOCKSTAT[SOCKSTAT["PID"].isin(MAPPING_PIDS)]
MAPPED_LSOF = LSOF[LSOF["PID"].isin(MAPPING_PIDS)]
fx.show(MAPPED_SOCKETS, n=len(MAPPED_SOCKETS),
        caption="Sockets attributed to object-mapping PIDs")
fx.show(MAPPED_LSOF, n=len(MAPPED_LSOF),
        caption="Open files attributed to object-mapping PIDs")
print(f"all sockets: {len(SOCKSTAT)}; attributed sockets: {len(MAPPED_SOCKETS)}; "
      f"all open files: {len(LSOF)}; attributed open files: {len(MAPPED_LSOF)}")

$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json linux.sockstat.Sockstat
# recorded: ok, exit 0
$ /home/anto/.local/bin/vol3 -f dumps/memory/mem.raw -s /home/anto/linux-multisource-dfir-lab/shared/isf -r json linux.lsof.Lsof
# recorded: ok, exit 0


**Full prepared socket inventory — showing 296 of 296 rows**

,Destination Addr,Destination Port,FD,Family,Filter,NetNS,PID,Process Name,Proto,Sock Offset,Source Addr,Source Port,State,TID,Type,parent_locator,locator
0,group:0x00000000,0,15,AF_NETLINK,"filter_type=socket_filter,bpf_filter_type=cBPF",4026531840,1,systemd,NETLINK_KOBJECT_UEVENT,155731286558720,groups:0x00000002,1,UNCONNECTED,1,RAW,None,sockstat.json#/0
1,NaN,NaN,16,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092120768,/run/systemd/notify,15644,UNCONNECTED,1,DGRAM,None,sockstat.json#/1
2,NaN,15646,17,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092113152,NaN,15645,CONNECTED,1,DGRAM,None,sockstat.json#/2
3,NaN,15645,18,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092114240,NaN,15646,CONNECTED,1,DGRAM,None,sockstat.json#/3
4,NaN,NaN,19,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092121856,/run/systemd/private,15647,LISTEN,1,STREAM,None,sockstat.json#/4
5,NaN,NaN,20,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092122944,/run/systemd/userdb/io.systemd.DynamicUser,15649,LISTEN,1,STREAM,None,sockstat.json#/5
6,NaN,NaN,21,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092118592,/run/systemd/io.system.ManagedOOM,15650,LISTEN,1,STREAM,None,sockstat.json#/6
7,NaN,20034,25,AF_UNIX,NaN,4026531840,1,systemd,NaN,155731268811648,/run/systemd/journal/stdout,20038,ESTABLISHED,1,STREAM,None,sockstat.json#/7
8,NaN,NaN,32,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092117504,/run/lvm/lvmpolld.socket,15658,LISTEN,1,STREAM,None,sockstat.json#/8
9,NaN,NaN,33,AF_UNIX,NaN,4026531840,1,systemd,NaN,155732092124032,,15660,LISTEN,1,STREAM,None,sockstat.json#/9


**Full prepared open-file inventory — showing 614 of 614 rows**

,Accessed,Changed,Device,FD,Inode,Mode,Modified,PID,Path,Process,Size,TID,Type,parent_locator,locator
0,2026-09-13T20:44:49.568000+00:00,2026-09-13T20:44:49.568000+00:00,0:5,0,5,crw-rw-rw-,2026-09-13T20:44:49.568000+00:00,1,/dev/null,systemd,0,1,CHR,None,lsof.json#/0
1,2026-09-13T20:44:49.568000+00:00,2026-09-13T20:44:49.568000+00:00,0:5,1,5,crw-rw-rw-,2026-09-13T20:44:49.568000+00:00,1,/dev/null,systemd,0,1,CHR,None,lsof.json#/1
2,2026-09-13T20:44:49.568000+00:00,2026-09-13T20:44:49.568000+00:00,0:5,2,5,crw-rw-rw-,2026-09-13T20:44:49.568000+00:00,1,/dev/null,systemd,0,1,CHR,None,lsof.json#/2
3,2026-09-13T20:44:49.560000+00:00,2026-09-13T20:44:49.560000+00:00,0:5,3,11,crw-r--r--,2026-09-13T20:44:49.560000+00:00,1,/dev/kmsg,systemd,0,1,CHR,None,lsof.json#/3
4,2026-09-13T20:44:45.419392+00:00,2026-09-13T20:44:45.419392+00:00,0:14,4,12487,?rw-------,2026-09-13T20:44:45.419392+00:00,1,anon_inode:[12487],systemd,0,1,NaN,None,lsof.json#/4
5,2026-09-13T20:44:45.419392+00:00,2026-09-13T20:44:45.419392+00:00,0:14,5,12487,?rw-------,2026-09-13T20:44:45.419392+00:00,1,anon_inode:[12487],systemd,0,1,NaN,None,lsof.json#/5
6,2026-09-13T20:44:45.419392+00:00,2026-09-13T20:44:45.419392+00:00,0:14,6,12487,?rw-------,2026-09-13T20:44:45.419392+00:00,1,anon_inode:[12487],systemd,0,1,NaN,None,lsof.json#/6
7,2026-09-13T20:44:48.940233+00:00,2026-09-13T20:44:49.544432+00:00,0:29,7,1,dr-xr-xr-x,2026-09-13T20:44:49.544432+00:00,1,/sys/fs/cgroup,systemd,0,1,DIR,None,lsof.json#/7
8,2026-09-13T20:44:45.419392+00:00,2026-09-13T20:44:45.419392+00:00,0:14,8,12487,?rw-------,2026-09-13T20:44:45.419392+00:00,1,anon_inode:[12487],systemd,0,1,NaN,None,lsof.json#/8
9,2026-09-13T20:44:45.419392+00:00,2026-09-13T20:44:45.419392+00:00,0:14,9,12487,?rw-------,2026-09-13T20:44:45.419392+00:00,1,anon_inode:[12487],systemd,0,1,NaN,None,lsof.json#/9


**Sockets attributed to object-mapping PIDs — showing 9 of 9 rows**

,Destination Addr,Destination Port,FD,Family,Filter,NetNS,PID,Process Name,Proto,Sock Offset,Source Addr,Source Port,State,TID,Type,parent_locator,locator
287,/run/systemd/journal/stdout,19332,1,AF_UNIX,NaN,4026531840,1087,sshd,NaN,155731281884992,NaN,20873,ESTABLISHED,1087,STREAM,None,sockstat.json#/287
288,/run/systemd/journal/stdout,19332,2,AF_UNIX,NaN,4026531840,1087,sshd,NaN,155731281884992,NaN,20873,ESTABLISHED,1087,STREAM,None,sockstat.json#/288
289,0.0.0.0,0,3,AF_INET,NaN,4026531840,1087,sshd,TCP,155731328641984,0.0.0.0,22,LISTEN,1087,STREAM,None,sockstat.json#/289
290,::,0,4,AF_INET6,NaN,4026531840,1087,sshd,TCP,155731284625792,::,22,LISTEN,1087,STREAM,None,sockstat.json#/290
291,192.168.100.1,54321,5,AF_INET,NaN,4026531840,1087,sshd,TCP,155731328648704,192.168.100.32,22,ESTABLISHED,1087,STREAM,None,sockstat.json#/291
292,192.168.100.1,54321,0,AF_INET,NaN,4026531840,1089,sh,TCP,155731328648704,192.168.100.32,22,ESTABLISHED,1089,STREAM,None,sockstat.json#/292
293,192.168.100.1,54321,1,AF_INET,NaN,4026531840,1089,sh,TCP,155731328648704,192.168.100.32,22,ESTABLISHED,1089,STREAM,None,sockstat.json#/293
294,192.168.100.1,54321,2,AF_INET,NaN,4026531840,1089,sh,TCP,155731328648704,192.168.100.32,22,ESTABLISHED,1089,STREAM,None,sockstat.json#/294
295,192.168.100.1,54321,5,AF_INET,NaN,4026531840,1089,sh,TCP,155731328648704,192.168.100.32,22,ESTABLISHED,1089,STREAM,None,sockstat.json#/295


**Open files attributed to object-mapping PIDs — showing 10 of 10 rows**

,Accessed,Changed,Device,FD,Inode,Mode,Modified,PID,Path,Process,Size,TID,Type,parent_locator,locator
604,2026-09-13T20:44:49.568000+00:00,2026-09-13T20:44:49.568000+00:00,0:5,0,5,crw-rw-rw-,2026-09-13T20:44:49.568000+00:00,1087,/dev/null,sshd,0,1087,CHR,None,lsof.json#/604
605,NaN,NaN,0:8,1,20873,srwxrwxrwx,NaN,1087,socket:[20873],sshd,0,1087,SOCK,None,lsof.json#/605
606,NaN,NaN,0:8,2,20873,srwxrwxrwx,NaN,1087,socket:[20873],sshd,0,1087,SOCK,None,lsof.json#/606
607,NaN,NaN,0:8,3,20882,srwxrwxrwx,NaN,1087,socket:[20882],sshd,0,1087,SOCK,None,lsof.json#/607
608,NaN,NaN,0:8,4,20884,srwxrwxrwx,NaN,1087,socket:[20884],sshd,0,1087,SOCK,None,lsof.json#/608
609,NaN,NaN,0:8,5,20902,srwxrwxrwx,NaN,1087,socket:[20902],sshd,0,1087,SOCK,None,lsof.json#/609
610,NaN,NaN,0:8,0,20902,srwxrwxrwx,NaN,1089,socket:[20902],sh,0,1089,SOCK,None,lsof.json#/610
611,NaN,NaN,0:8,1,20902,srwxrwxrwx,NaN,1089,socket:[20902],sh,0,1089,SOCK,None,lsof.json#/611
612,NaN,NaN,0:8,2,20902,srwxrwxrwx,NaN,1089,socket:[20902],sh,0,1089,SOCK,None,lsof.json#/612
613,NaN,NaN,0:8,5,20902,srwxrwxrwx,NaN,1089,socket:[20902],sh,0,1089,SOCK,None,lsof.json#/613


all sockets: 296; attributed sockets: 9; all open files: 614; attributed open files: 10


**Interpretation.** _(to write)_

### 3.6 Does memory hold shell history the disk does not?

Section 2.4 found zero allocated history files in its stated scope. Shell PIDs are selected from
the process inventory and restricted to processes that mapped the Section 1 object.

In [23]:
shell_mask = PSAUX["COMM"].astype("string").str.match(r"(?:ba)?sh$", na=False)
SHELL_ROWS = PSAUX[shell_mask & PSAUX["PID"].isin(MAPPING_PIDS)]
fx.show(SHELL_ROWS, cols=("PID", "PPID", "COMM", "ARGS", "locator"),
        n=20, caption="Object-mapping shell processes")
SHELL_PIDS = SHELL_ROWS["PID"].dropna().astype(int).tolist()
print(f"shell PIDs selected from 3.2 and 3.3: {SHELL_PIDS}")
BASH = pd.DataFrame()
if SHELL_PIDS:
    pid_args = " ".join(fx.q(pid) for pid in SHELL_PIDS)
    bash_json = OUT / "s3-06-bash.json"
    bash_run = fx.sh(f"{VOL_BASE} linux.bash.Bash --pid {pid_args} > {fx.q(bash_json)}",
                     label="s3-06-bash-run", out_dir=OUT, tail=20)
    print(f"linux.bash exit {bash_run.returncode}")
    if bash_run.returncode == 0:
        BASH = fx.load_vol(bash_json)
        fx.show(BASH, n=100, caption="Recovered in-memory shell history")
        print(f"recovered shell-history rows: {len(BASH)}")
else:
    print("linux.bash not run: zero relevant shell PIDs")

**Object-mapping shell processes — showing 1 of 1 rows**

,PID,PPID,COMM,ARGS,locator
128,1089,1087,sh,/bin/sh,psaux.json#/128


shell PIDs selected from 3.2 and 3.3: [1089]
$ vol3 -f dumps/memory/mem.raw -s ../../isf -r json linux.bash.Bash --pid 1089 > investigation/output/s3-06-bash.json



Progress:   43.36		Scanning Elf64Layer using BytesScanner

Progress:   43.75		Scanning Elf64Layer using BytesScanner

Progress:   44.14		Scanning Elf64Layer using BytesScanner

Progress:   44.53		Scanning Elf64Layer using BytesScanner

Progress:   44.92		Scanning Elf64Layer using BytesScanner

Progress:   45.31		Scanning Elf64Layer using BytesScanner

Progress:   45.70		Scanning Elf64Layer using BytesScanner

Progress:   46.09		Scanning Elf64Layer using BytesScanner

Progress:   46.48		Scanning Elf64Layer using BytesScanner

Progress:  100.00		Stacking attempts finished            


linux.bash exit 0


**Recovered in-memory shell history — showing 0 of 0 rows**

""


recovered shell-history rows: 0


**Interpretation.** _(to write)_

### 3.7 Can the object be recovered from memory?

The lowest PID in the enumerated mapping pairs is the reproducible selection rule. SHA-256 is
compared with the Section 1 disk extraction; equality and inequality are both meaningful.

In [24]:
ELF_PID = min(MAPPING_PIDS, default=None)
ELF_DIR = DATA / "s3-07-elfs"
ELF_DIR.mkdir(parents=True, exist_ok=True)
ELFS = pd.DataFrame()
DUMPED_OBJECTS = []
print(f"mapping PID selected for linux.elfs: {ELF_PID}")
if ELF_PID is not None:
    elfs_json = OUT / "s3-07-elfs.json"
    elf_cmd = (f"vol3 -o {fx.q(ELF_DIR)} -f {fx.q(MEM)} -s {fx.q(ISF_DIR)} "
               f"-r json linux.elfs.Elfs --pid {fx.q(ELF_PID)} --dump")
    elf_run = fx.sh(f"{elf_cmd} > {fx.q(elfs_json)}",
                    label="s3-07-elfs-run", out_dir=OUT, tail=20)
    print(f"linux.elfs exit {elf_run.returncode}")
    if elf_run.returncode == 0:
        ELFS = fx.load_vol(elfs_json)
        object_elfs = ELFS[ELFS["File Path"].eq(OBJECT_MAP_PATH)]
        fx.show(object_elfs, n=40, caption="Recovered rows for the mapped object")
        outputs = object_elfs.get("File Output", pd.Series(dtype="string"))
        DUMPED_OBJECTS = [ELF_DIR / str(name) for name in outputs if pd.notna(name)]
        print(f"object dump files present: {len(DUMPED_OBJECTS)}")
        for index, dumped in enumerate(DUMPED_OBJECTS, 1):
            fx.sh(f"sha256sum {fx.q(OBJECT_BIN)} {fx.q(dumped)}",
                  label=f"s3-07-comparison-{index}", out_dir=OUT)
            print(f"byte-identical to disk object: "
                  f"{fx.sha256_file(OBJECT_BIN) == fx.sha256_file(dumped)}")

mapping PID selected for linux.elfs: 1087
$ vol3 -o investigation/data/s3-07-elfs -f dumps/memory/mem.raw -s ../../isf -r json linux.elfs.Elfs --pid 1087 --dump > investigation/output/s3-07-elfs.json



Progress:   43.36		Scanning Elf64Layer using BytesScanner

Progress:   43.75		Scanning Elf64Layer using BytesScanner

Progress:   44.14		Scanning Elf64Layer using BytesScanner

Progress:   44.53		Scanning Elf64Layer using BytesScanner

Progress:   44.92		Scanning Elf64Layer using BytesScanner

Progress:   45.31		Scanning Elf64Layer using BytesScanner

Progress:   45.70		Scanning Elf64Layer using BytesScanner

Progress:   46.09		Scanning Elf64Layer using BytesScanner

Progress:   46.48		Scanning Elf64Layer using BytesScanner

Progress:  100.00		Stacking attempts finished            


linux.elfs exit 0


**Recovered rows for the mapped object — showing 1 of 1 rows**

,End,File Output,File Path,PID,Process,Start,parent_locator,locator
27,140178528174080,pid.1087.sshd.0x7f7ddb606000-14.dmp,/usr/lib/selinux.so.3,1087,sshd,140178528165888,None,s3-07-elfs.json#/27


object dump files present: 1
$ sha256sum investigation/data/s1-05-referenced-object.bin investigation/data/s3-07-elfs/pid.1087.sshd.0x7f7ddb606000-14.dmp
87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711  investigation/data/s1-05-referenced-object.bin
95d79b37ec7f673ed32ce97b3f41bf79725c769babd31717ec4f1703a9fef506  investigation/data/s3-07-elfs/pid.1087.sshd.0x7f7ddb606000-14.dmp
byte-identical to disk object: False


**Interpretation.** _(to write)_

### 3.8 Are kernel-level mechanisms clean?

Valid zero rows from the inline kernel checks are reported as results, not failures.

In [25]:
DEBUG=True
if (not DEBUG): # disable this cell temporarily to rerun the notebook faster
    LSMOD = fx.load_vol(product("lsmod.json"))
    KMSG = fx.load_vol(product("kmsg.json"))
    fx.show(LSMOD, n=len(LSMOD), caption="Full prepared module inventory")
    print(f"prepared kmsg rows: {len(KMSG)}; displaying the final 80")
    fx.show(KMSG.tail(80), n=80, caption="Final prepared kernel-log rows")
    KERNEL_CHECKS = {}
    plugins = (
        ("linux.malware.check_syscall.Check_syscall", "check-syscall"),
        ("linux.malware.modxview.Modxview", "modxview"),
        ("linux.ebpf.EBPF", "ebpf"),
    )
    for plugin_name, label in plugins:
        target = OUT / f"s3-08-{label}.json"
        run = fx.sh(f"{VOL_BASE} {plugin_name} > {fx.q(target)}",
                    label=f"s3-08-{label}-run", out_dir=OUT, tail=20)
        print(f"{plugin_name} exit {run.returncode}")
        if run.returncode != 0:
            continue
        result = fx.load_vol(target)
        KERNEL_CHECKS[label] = result
        print(f"{label} rows: {len(result)}")
        fx.show(result, n=80, caption=f"{plugin_name} result")

**Interpretation.** _(to write)_

## 4. Deletion recovery

### 4.1 Do deleted directory entries survive under `/tmp`?

The target `/tmp/rk.so` comes from the journal lines recovered in 2.3. Show only regular-file
rows in the deleted directory listing, then count the unallocated inode records and zero sizes.
Ext4 clears the size and extent tree on unlink; ordinary undelete cannot use that cleared map.
A missing directory entry is a result bounded to this traversal.

In [26]:
TARGET_PATH = "/tmp/rk.so"
TSK = f"-i ewf -b {fx.q(SECTOR_SIZE)} -f ext4 -o {fx.q(ROOT_OFFSET)}"
METADATA_FAILED = False
fls_deleted = ils = None
try:
    TMP_INODE = str(ALLOCATED.loc[ALLOCATED["name"].eq("/tmp"), "inode"].item())
    fls_deleted = fx.sh(f"fls {TSK} -rd -p {fx.q(IMG)} {fx.q(TMP_INODE)}",
                        label="s4-01-tmp-deleted", out_dir=OUT, check=False, tail=0)
    regular_rows = [line for line in fls_deleted.stdout.splitlines() if line.startswith("r/")]
    print("\n".join(regular_rows) or "0 deleted regular-file rows under /tmp")
    ils = fx.sh(f"ils {TSK} -A -Z {fx.q(IMG)}", label="s4-01-ils",
                out_dir=OUT, check=False, tail=0)
    METADATA_FAILED = fls_deleted.returncode != 0 or ils.returncode != 0
    if not METADATA_FAILED:
        inode_rows = [line.split("|") for line in ils.stdout.splitlines()
                      if line.split("|")[0].isdigit()]
        zero_sizes = sum(int(row[10]) == 0 for row in inode_rows)
        print(f"unallocated inode records: {len(inode_rows)}; size 0: {zero_sizes}")
    else:
        print(f"tool failure: fls exit {fls_deleted.returncode}; ils exit {ils.returncode}")
except (ValueError, IndexError, OSError) as error:
    METADATA_FAILED = True
    print(f"tool failure: {error}")

$ fls -i ewf -b 512 -f ext4 -o 227328 -rd -p dumps/disk/evidence_disk.E01 1581
0 deleted regular-file rows under /tmp
$ ils -i ewf -b 512 -f ext4 -o 227328 -A -Z dumps/disk/evidence_disk.E01
unallocated inode records: 33; size 0: 33


**Interpretation.** _(to write)_

### 4.2 How large is the journal, and can it be exported intact?

Read the size of journal inode 8, then export only that inode to `investigation/data/journal.bin`.

In [27]:
journal_istat = fx.sh(f"istat {TSK} {fx.q(IMG)} 8", label="s4-02-journal-istat",
                       out_dir=OUT, check=False, tail=0)
JOURNAL_BIN = DATA / "journal.bin"
JOURNAL_FAILED = journal_istat.returncode != 0
JOURNAL_EXPORTED = False
try:
    JOURNAL_SIZE = int(re.search(r"^size:\s*(\d+)", journal_istat.stdout, re.M | re.I)[1])
    JOURNAL_BLOCKS = JOURNAL_SIZE // 4096
    journal_export = fx.sh(f"icat {TSK} {fx.q(IMG)} 8 > {fx.q(JOURNAL_BIN)}",
                           label="s4-02-journal-icat", out_dir=OUT, check=False)
    exported_size = JOURNAL_BIN.stat().st_size
    JOURNAL_EXPORTED = (journal_export.returncode == 0 and exported_size == JOURNAL_SIZE
                        and JOURNAL_SIZE > 0 and JOURNAL_SIZE % 4096 == 0)
    JOURNAL_FAILED |= not JOURNAL_EXPORTED
    print(f"istat length: {JOURNAL_SIZE} bytes; {JOURNAL_BLOCKS} blocks of 4096 bytes")
    print(f"export: {exported_size} bytes; icat exit {journal_export.returncode}")
except (TypeError, ValueError, OSError) as error:
    JOURNAL_FAILED = True
    print(f"tool failure: {error}")

$ istat -i ewf -b 512 -f ext4 -o 227328 dumps/disk/evidence_disk.E01 8
$ icat -i ewf -b 512 -f ext4 -o 227328 dumps/disk/evidence_disk.E01 8 > investigation/data/journal.bin
istat length: 67108864 bytes; 16384 blocks of 4096 bytes
export: 67108864 bytes; icat exit 0


**Interpretation.** _(to write)_

### 4.3 Do old directory blocks in the journal name the file?

Traverse the export from the data directory with a 120-second limit. Stdout and stderr are
preserved separately. `logdump: short read (read 0, expected 4096)` is the reader stepping one
block past the export; it is an expected terminal condition when the traversal completes.

In [28]:
import struct
DEBUGFS_RUN = byte_hits = None
JOURNAL_BYTES = b""
DIR_INODES = set()
if JOURNAL_EXPORTED:
    request = f"logdump -O -a -n {JOURNAL_BLOCKS} -f journal.bin"
    command = f"cd {fx.q(DATA)} && timeout -k 5s 120s debugfs -R {fx.q(request)}"
    DEBUGFS_RUN = fx.sh(command, label="s4-03-journal-index", out_dir=OUT,
                        check=False, tail=0)
    terminal = "logdump: short read (read 0, expected 4096)"
    unexpected = [s for s in DEBUGFS_RUN.stderr.splitlines()
                  if s and not s.startswith("debugfs ") and not s.startswith(terminal)]
    JOURNAL_FAILED |= DEBUGFS_RUN.returncode != 0 or bool(unexpected)
    print(f"debugfs exit {DEBUGFS_RUN.returncode}; traversal completed: {not JOURNAL_FAILED}")
    print("stderr:\n" + DEBUGFS_RUN.stderr.rstrip())
    byte_hits = fx.sh(f"grep -aboF {fx.q(Path(TARGET_PATH).name)} {fx.q(JOURNAL_BIN)}",
                      label="s4-03-journal-name-hits", out_dir=OUT, check=False)
    JOURNAL_FAILED |= byte_hits.returncode not in (0, 1)
else:
    print("not attempted: intact journal export unavailable")

$ cd investigation/data && timeout -k 5s 120s debugfs -R 'logdump -O -a -n 16384 -f journal.bin'
debugfs exit 0; traversal completed: True
stderr:
debugfs 1.47.0 (5-Feb-2023)
logdump: short read (read 0, expected 4096) while reading journal
$ grep -aboF rk.so investigation/data/journal.bin
3707244:rk.so
3834220:rk.so
3998060:rk.so
4067692:rk.so
4198764:rk.so


Decode each name hit as an `ext4_dir_entry_2`, including the preceding entry in the same block.
The printed fields are inode, rec_len, name_len, file_type, name and journal block. Compare the
inode against the objects resolved or inventoried in Sections 1–2, including the staged files.

In [29]:
try:
    JOURNAL_BYTES = JOURNAL_BIN.read_bytes() if JOURNAL_EXPORTED else b""
    for hit in byte_hits.stdout.splitlines() if byte_hits is not None else []:
        offset = int(hit.split(":", 1)[0])
        start, block_start = offset - 8, offset // 4096 * 4096
        inode, rec_len, name_len, file_type = struct.unpack_from("<IHBB", JOURNAL_BYTES, start)
        name = JOURNAL_BYTES[offset:offset + name_len].decode()
        valid_name = name_len == len(Path(TARGET_PATH).name) and name == Path(TARGET_PATH).name
        print(inode, rec_len, name_len, file_type, name, offset // 4096,
              f"name_len valid: {valid_name}")
        if not valid_name:
            raise ValueError("directory name length/content mismatch")
        DIR_INODES.add(inode)
        cursor, previous = block_start, None
        while cursor < start:
            pi, pr, pn, pt = struct.unpack_from("<IHBB", JOURNAL_BYTES, cursor)
            if pr < 8 or cursor + pr > block_start + 4096:
                raise ValueError("invalid preceding directory entry length")
            previous = (pi, pr, pn, pt, JOURNAL_BYTES[cursor + 8:cursor + 8 + pn].decode())
            cursor += pr
        print("preceding entry:", previous)
except (OSError, ValueError, UnicodeError, struct.error) as error:
    JOURNAL_FAILED = True
    print(f"tool failure: {error}")

74252 56 5 1 rk.so 905 name_len valid: True
preceding entry: (74173, 28, 17, 1, '__malicious_recon')
74252 56 5 1 rk.so 936 name_len valid: True
preceding entry: (74173, 28, 17, 1, '__malicious_recon')
74252 56 5 1 rk.so 976 name_len valid: True
preceding entry: (74173, 28, 17, 1, '__malicious_recon')
74252 16 5 1 rk.so 993 name_len valid: True
preceding entry: (74173, 28, 17, 1, '__malicious_recon')
74252 16 5 1 rk.so 1025 name_len valid: True
preceding entry: (74173, 28, 17, 1, '__malicious_recon')


In [30]:
RECOVERED_INODE = next(iter(DIR_INODES)) if len(DIR_INODES) == 1 else None
versions = {}
known_inodes = {int(preload_inode), int(object_inode), int(shadow_inode)}
known_inodes.update(int(entry["inode"]) for entry in object_chain)
known_inodes.update(int(item["inode"]) for item in staged)
for rows in (log_rows, sudo_rows):
    known_inodes.update(int(row.inode) for row in rows.itertuples())
for path in ACCOUNT_FILES:
    known_inodes.update(int(v) for v in ALLOCATED.loc[ALLOCATED["name"].eq(path), "inode"])
print(f"recovered inode: {RECOVERED_INODE}; matches Sections 1–2: "
      f"{RECOVERED_INODE in known_inodes}; matching inodes: {sorted(DIR_INODES & known_inodes)}")

recovered inode: 74252; matches Sections 1–2: True; matching inodes: [74252]


**Interpretation.** _(to write)_

### 4.4 Does the journal hold pre-unlink copies of the inode itself?

Read the group geometry from the prepared fsstat. Compute the inode-table block and byte offset,
then decode every logged copy. Versions are deduplicated and ordered by their first journal
block, not an assumed wall-clock chronology. Each tuple prints **mode, size, links, mtime, dtime,
extents**; timestamps are raw Unix seconds. Each extent is **logical block, length, physical
block, initialized**. Only an extent header with magic `0xF30A` and depth zero supplies a map.

In [37]:
if RECOVERED_INODE is not None:
    try:
        fs = Path(prepared["products"][ROOT_FS_PRODUCT]["path"]).read_text()
        ipg, isize, bsize = [int(re.search(p, fs)[1]) for p in (
            r"Inodes per group:\s*(\d+)", r"Inode Size:\s*(\d+)", r"Block Size:\s*(\d+)")]
        group, index = divmod(RECOVERED_INODE - 1, ipg)
        table = int(re.search(rf"Group: {group}:.*?Inode Table: (\d+)", fs, re.S)[1])
        fs_block, offset_in_bl = table + index * isize // bsize, index * isize % bsize
        for jb in sorted(set(map(int, re.findall(
                rf"FS block {fs_block} logged at journal block (\d+)", DEBUGFS_RUN.stdout)))):
            raw = JOURNAL_BYTES[jb * bsize + offset_in_bl:][:isize]
            mode, uid, size, atime, ctime, mtime, dtime = struct.unpack_from("<HHIIIII", raw)
            magic, count, maximum, depth = struct.unpack_from("<HHHH", raw, 40)
            count = count if (magic, depth) == (0xF30A, 0) else 0
            extents = [struct.unpack_from("<IHHI", raw, 52 + 12*k) for k in range(count)]
            extents = tuple((a, n - 32768 if n > 32768 else n, hi << 32 | lo, n <= 32768)
                            for a, n, hi, lo in extents)
            version = (mode, size, struct.unpack_from("<H", raw, 26)[0], mtime, dtime, extents)
            versions.setdefault(version, []).append(jb)
    except (OSError, TypeError, ValueError, AttributeError, struct.error) as error:
        JOURNAL_FAILED = True
        print(f"tool failure: {error}")
for version, copies in versions.items():
    print(f"first journal block {copies[0]}, copies {len(copies)}: {version}")

first journal block 308, copies 4: (17407, 0, 0, 1786468439, 1786468439, ())
first journal block 882, copies 2: (33152, 0, 0, 1789332301, 1789332301, ())
first journal block 903, copies 2: (33204, 32784, 1, 1789332312, 0, ((0, 9, 338469, False),))
first journal block 909, copies 10: (33204, 32784, 1, 1789332312, 0, ((0, 9, 338469, True),))
first journal block 1043, copies 2: (33204, 0, 0, 1789332390, 1789332390, ())
first journal block 1109, copies 2: (33188, 17, 1, 1789332392, 0, ((0, 1, 296191, True),))


**Interpretation.** _(to write)_

### 4.5 Can content be recovered from the old extent map?

Take the first version with non-zero size and initialized extents. `blkcat` reads its physical
blocks from the acquired **image**, places them at their logical offsets, and trims to the old
size. These blocks may have been reallocated between unlink and imaging. A hash mismatch is
`partial content recovered` or `no result in examined scope`; the name, size, timestamps and
block map recovered in 4.3–4.4 remain available either way.

In [38]:
RECOVERED = DATA / "s4-05-recovered-object.bin"
chosen = next((v for v in versions if v[1] > 0 and v[5]
               and all(length > 0 and initialized for _, length, _, initialized in v[5])), None)
RECOVERED_BYTES = 0
if chosen is not None:
    print(f"selected version: first journal block {versions[chosen][0]}; size {chosen[1]}")
    try:
        with RECOVERED.open("wb") as recovered:
            for logical, length, physical, initialized in chosen[5]:
                part = DATA / f"s4-05-extent-{logical}.bin"
                command = (f"blkcat {TSK} {fx.q(IMG)} {fx.q(physical)} {fx.q(length)} "
                           f"> {fx.q(part)}")
                read = fx.sh(command, label=f"s4-05-extent-{logical}",
                             out_dir=OUT, check=False)
                content = part.read_bytes()
                JOURNAL_FAILED |= read.returncode != 0 or len(content) != length * bsize
                recovered.seek(logical * bsize)
                RECOVERED_BYTES += recovered.write(content)
            recovered.truncate(chosen[1])
    except OSError as error:
        JOURNAL_FAILED = True
        print(f"tool failure: {error}")
else:
    print("not attempted: no non-zero version with initialized extents")

selected version: first journal block 909; size 32784
$ blkcat -i ewf -b 512 -f ext4 -o 227328 dumps/disk/evidence_disk.E01 338469 9 > investigation/data/s4-05-extent-0.bin


In [33]:
REFERENCE_HASH = RECOVERED_HASH = None
CONTENT_MATCH = False
try:
    REFERENCE_HASH = fx.sha256_file(OBJECT_BIN)
    if chosen is not None and RECOVERED_BYTES:
        identified = fx.sh(f"file {fx.q(RECOVERED)}", label="s4-05-file",
                           out_dir=OUT, check=False)
        hashed = fx.sh(f"sha256sum {fx.q(OBJECT_BIN)} {fx.q(RECOVERED)}",
                       label="s4-05-sha256", out_dir=OUT, check=False)
        JOURNAL_FAILED |= identified.returncode != 0 or hashed.returncode != 0
        RECOVERED_HASH = fx.sha256_file(RECOVERED)
        CONTENT_MATCH = RECOVERED_HASH == REFERENCE_HASH
    print(f"Section 1.5 SHA-256: {REFERENCE_HASH}")
    print(f"recovered SHA-256:   {RECOVERED_HASH}; match: {CONTENT_MATCH}")
except OSError as error:
    JOURNAL_FAILED = True
    print(f"tool failure: {error}")

$ file investigation/data/s4-05-recovered-object.bin
investigation/data/s4-05-recovered-object.bin: ELF 64-bit LSB shared object, x86-64, version 1 (SYSV), dynamically linked, BuildID[sha1]=96daef8bb7ab389abcb5aa9458436759949849c7, not stripped
$ sha256sum investigation/data/s1-05-referenced-object.bin investigation/data/s4-05-recovered-object.bin
87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711  investigation/data/s1-05-referenced-object.bin
87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711  investigation/data/s4-05-recovered-object.bin
Section 1.5 SHA-256: 87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711
recovered SHA-256:   87fece49fc15a48372a1ba76cf424755f9cfab6cce7e8073002757f7db2f0711; match: True


**Interpretation.** _(to write)_

### 4.6 Does free-space ELF carving recover matching content?

Print the installed version, then obtain PhotoRec's partition listing without starting recovery.
Select the unique Linux filesystem from that listing; its partition number is PhotoRec's own,
not the TSK sector offset. Preserve the listing and both command streams in log files.

In [34]:
import tempfile
photorec_version = fx.sh("photorec --version", label="s4-06-version",
                         out_dir=OUT, check=False)
CARVE_FAILED = photorec_version.returncode != 0
PHOTOREC_PARTITION = None
PHOTOREC_DIR = Path(os.path.relpath(tempfile.mkdtemp(prefix="s4-06-photorec-", dir=OUT)))
LIST_LOG = PHOTOREC_DIR / "partitions.log"
command = (f"timeout -k 5s 120s photorec /log /logname {fx.q(LIST_LOG)} "
           f"/cmd {fx.q(IMG)} '' </dev/null > {fx.q(PHOTOREC_DIR / 'listing.txt')} 2>&1")
listing = fx.sh(command, check=False)
try:
    listing_text = LIST_LOG.read_text()
    partitions = re.findall(r"^\s*(\d+) P Linux filesys\. data\s+", listing_text, re.M)
    CARVE_FAILED |= listing.returncode != 0 or "PhotoRec exited normally." not in listing_text
    if len(partitions) == 1 and not CARVE_FAILED:
        PHOTOREC_PARTITION = partitions[0]
    else:
        CARVE_FAILED = True
except OSError as error:
    CARVE_FAILED = True
    print(f"tool failure: {error}")

$ photorec --version
PhotoRec 7.2, Data Recovery Utility, February 2024
Christophe GRENIER <grenier@cgsecurity.org>
https://www.cgsecurity.org

Version: 7.2
Compiler: GCC 4.8
ext2fs lib: 1.42.8, ntfs lib: libntfs-3g, ewf lib: 20120504, libjpeg: libjpeg-turbo-1.3.1, curses lib: ncurses 5.9, zlib: 1.2.7
iconv support: no
OS: Linux, kernel 6.8.0-139-generic (#139-Ubuntu SMP PREEMPT_DYNAMIC Sat Aug  1 03:52:05 UTC 2026) x86_64
$ timeout -k 5s 120s photorec /log /logname investigation/output/s4-06-photorec-3d2h7e5b/partitions.log /cmd dumps/disk/evidence_disk.E01 '' </dev/null > investigation/output/s4-06-photorec-3d2h7e5b/listing.txt 2>&1


Run only ELF signature carving in that partition's free space, directly against the image.
A fresh output directory keeps reruns distinct. The progress stream goes to a log file; report
only the exit status, candidate count, output bytes and hash comparison. A matching carved
object alone does not establish its former pathname.

In [35]:
PHOTOREC_RUN, CANDIDATES, MATCHING_CANDIDATES = None, [], []
TOTAL_CARVED_BYTES = 0
if PHOTOREC_PARTITION is not None:
    selection = (f"{PHOTOREC_PARTITION},options,mode_ext2,fileopt,everything,disable,"
                 "elf,enable,freespace,search")
    command = (f"timeout -k 5s 1200s photorec /log "
               f"/logname {fx.q(PHOTOREC_DIR / 'photorec.log')} "
               f"/d {fx.q(PHOTOREC_DIR / 'files')} /cmd {fx.q(IMG)} {fx.q(selection)} "
               f"> {fx.q(PHOTOREC_DIR / 'console.log')} 2>&1")
    PHOTOREC_RUN = fx.sh(command, check=False)
    try:
        log = (PHOTOREC_DIR / "photorec.log").read_text()
        CARVE_FAILED |= (PHOTOREC_RUN.returncode != 0 or "Total: " not in log
                         or "PhotoRec exited normally." not in log)
        CANDIDATES = sorted(PHOTOREC_DIR.glob("files.*/*.elf"))
        TOTAL_CARVED_BYTES = sum(p.stat().st_size for p in CANDIDATES)
        MATCHING_CANDIDATES = [p for p in CANDIDATES if fx.sha256_file(p) == REFERENCE_HASH]
    except OSError as error:
        CARVE_FAILED = True
        print(f"tool failure: {error}")
print(f"exit: {PHOTOREC_RUN.returncode if PHOTOREC_RUN else 'not attempted'}; candidates: "
      f"{len(CANDIDATES)}; bytes: {TOTAL_CARVED_BYTES}; hash match: {bool(MATCHING_CANDIDATES)}")

$ timeout -k 5s 1200s photorec /log /logname investigation/output/s4-06-photorec-3d2h7e5b/photorec.log /d investigation/output/s4-06-photorec-3d2h7e5b/files /cmd dumps/disk/evidence_disk.E01 1,options,mode_ext2,fileopt,everything,disable,elf,enable,freespace,search > investigation/output/s4-06-photorec-3d2h7e5b/console.log 2>&1
exit: 0; candidates: 4196; bytes: 798285824; hash match: False


**Interpretation.** _(to write)_

### 4.7 What did each technique return?

These are recovery outcomes within the examined scope, not accepted findings or evidence of
erasure. Raw command streams and the executed notebook retain the observations for review.

In [36]:
a_outcome = ("tool failure" if METADATA_FAILED else
             "metadata or name trace only" if Path(TARGET_PATH).name in fls_deleted.stdout else
             "no result in examined scope")
b_outcome = ("tool failure" if JOURNAL_FAILED else
             "content recovered" if CONTENT_MATCH else
             "partial content recovered" if RECOVERED_BYTES else
             "metadata or name trace only" if DIR_INODES or versions else
             "no result in examined scope")
c_outcome = ("tool failure" if CARVE_FAILED else
             "not attempted" if PHOTOREC_RUN is None else
             "content recovered" if MATCHING_CANDIDATES else
             "no result in examined scope")
RECOVERY_OUTCOMES = pd.DataFrame([
    ("Surviving metadata", a_outcome, "investigation/output/s4-01-*.txt"),
    ("Journal-assisted reconstruction", b_outcome,
     "investigation/data/journal.bin; investigation/output/s4-0[3-5]-*.txt"),
    ("Free-space ELF carving", c_outcome, str(PHOTOREC_DIR)),
], columns=["technique", "outcome", "raw output"])
print(RECOVERY_OUTCOMES.to_string(index=False))

                      technique                     outcome                                                           raw output
             Surviving metadata no result in examined scope                                     investigation/output/s4-01-*.txt
Journal-assisted reconstruction           content recovered investigation/data/journal.bin; investigation/output/s4-0[3-5]-*.txt
         Free-space ELF carving no result in examined scope                         investigation/output/s4-06-photorec-3d2h7e5b


**Interpretation.** _(to write)_

## 5. Chronology

`mactime` over the allocated bodyfile and the Plaso export, both scoped to the incident window,
merged into one sourced chronology. Acquisition and examiner artifacts get their own lane.

**Not implemented. Plaso required.**

## 6. Result tables

Findings are written here, by hand, from the observations above. Then the four tables and the
permitted counts, per [ai/RULES.md](../../ai/RULES.md).

**Not implemented.**

## 7. Validation against the controlled scenario

Compare the reconstruction with the run's command log: agreements, discrepancies, and what the
evidence could not show. Ground truth is an experimental reference, not a fourth source.

**Not implemented.**